# 疑似ラベルによる BIO tagging 事前学習

`create_pseudo_dataset.py` が生成する JSONL を入力として、
テキストからスキル・趣味のスパンを抽出するモデルを疑似ラベルで事前学習する。

## Google Drive マウント

入力データ（JSONL）と学習済みモデルの保存先を Google Drive に設定し、セッション終了後も永続化する。

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 依存パッケージのインストール

In [4]:
!pip install datasets pydantic transformers seqeval fugashi unidic-lite wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 44.3 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 694.9/694.9 kB 55.1 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=fb2453f02bc5f7d0e93ee39bf3e409d09fa3fc827baa9be0f06456105a1c0ace
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
  Created wheel for unidic-lite: filename=unidic_lite-1.0.8-py3-none-any.whl size=47658817 sha256=7395eab8f537d95b51db421e7f3b5c01f72e630c1d2e6177ee1eb117fa36eed7
  Stored in directory: /root/.cache/pip/wheels/5e/1f/0f/4d43887e5476d956fae828ee9b6687becd5544d68b51ed633d
Successfully built seqeval unidic-lite


## インポート

In [5]:
import logging
from collections import defaultdict
from collections.abc import Callable

import numpy as np
import numpy.typing as npt
from datasets import Dataset, DatasetDict, load_dataset
from pydantic.dataclasses import dataclass
from seqeval.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    EvalPrediction,
    PreTrainedTokenizerBase,
    Trainer,
    TrainingArguments,
)
from transformers.modeling_utils import PreTrainedModel
from transformers.tokenization_utils_base import BatchEncoding

import wandb

In [6]:
from google.colab import userdata

## ロガー設定

In [7]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## BIO ラベル定義

In [8]:
LABEL_NAMES: tuple[str, ...] = ("NONE", "PROFILE", "PII")
LABEL2ID: dict[str, int] = {"NONE": 0, "PROFILE": 1, "PII": 2}
ID2LABEL: dict[int, str] = {0: "NONE", 1: "PROFILE", 2: "PII"}

## Config

In [9]:
@dataclass(frozen=True)
class Config:
    """疑似事前学習の設定。

    Attributes:
        exp: 実験番号。wandb の run name と出力先ディレクトリに使用する。
        model_name: ベースモデル名。
        data_path: 疑似ラベル JSONL のパス。
        output_base_dir: モデル出力のベースディレクトリ。
            実際の出力先は ``{output_base_dir}/{exp}`` になる。
        wandb_project: wandb のプロジェクト名。
        max_length: トークン最大長。
        num_train_epochs: 学習エポック数。
        per_device_train_batch_size: デバイスあたりのバッチサイズ。
        gradient_accumulation_steps: 勾配累積ステップ数。
        learning_rate: 学習率。
        weight_decay: 重み減衰。
        warmup_ratio: 学習率ウォームアップの割合。
        fp16: FP16 混合精度学習を有効にするか。
        dataloader_num_workers: DataLoader のワーカー数。
        eval_ratio: 評価データの割合。
        seed: 乱数シード。
        logging_steps: ログ出力間隔（ステップ数）。
        eval_strategy: 評価戦略。\"steps\" または \"epoch\"。
        save_strategy: チェックポイント保存戦略。\"steps\" または \"epoch\"。
        eval_steps: 評価間隔（ステップ数）。eval_strategy が \"steps\" の場合に使用。
        save_steps: チェックポイント保存間隔（ステップ数）。save_strategy が \"steps\" の場合に使用。
        save_total_limit: 保存するチェックポイントの最大数。
        metric_for_best_model: ベストモデル判定に使うメトリクス名。
        report_to: 学習メトリクスのレポート先。
        early_stopping_patience: Early stopping の patience。
    """

    exp: str = "0020"
    model_name: str = "sbintuitions/modernbert-ja-130m"
    data_path: str = "drive/My Drive/Colab Notebooks/quasi_identifer/annotations_1200.jsonl"
    output_base_dir: str = "drive/My Drive/Colab Notebooks/quasi_identifer"
    wandb_project: str = "pseudo-sft"
    max_length: int = 128
    num_train_epochs: int = 2
    per_device_train_batch_size: int = 16
    gradient_accumulation_steps: int = 2
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    fp16: bool = True
    dataloader_num_workers: int = 4
    eval_ratio: float = 0.1
    seed: int = 42
    logging_steps: int = 100
    eval_strategy: str = "steps"
    save_strategy: str = "steps"
    eval_steps: int = 100
    save_steps: int = 100
    save_total_limit: int = 1
    metric_for_best_model: str = "f1"
    report_to: str = "wandb"
    early_stopping_patience: int = 1000

## データ読み込み関数

In [10]:
def load_pseudo_dataset(data_path: str) -> Dataset:
    """疑似ラベル JSONL を読み込み、グループ済み Dataset を返す。

    現行の create_pseudo_dataset.py が出力する pseudo_labels（リスト）形式を
    主形式として扱う。旧形式（pseudo_label 単数、1ラベル1行）の JSONL にも
    後方互換として対応し、同一テキストでグループ化してリストに集約する。

    Args:
        data_path: JSONL ファイルパス。

    Returns:
        text と pseudo_labels カラムを持つ Dataset。
    """
    ds = load_dataset("json", data_files=data_path, split="train")
    logger.info("loaded %d rows from %s", len(ds), data_path)

    # 同じ内容が重複している場合はスキップ
    grouped_ds = []
    phrases = set([])
    for row in ds:
        if row["clause"] in phrases:
            continue
        phrases.add(row["clause"])
        row["text"] = row["clause"]
        grouped_ds.append(row)
    grouped_ds: Dataset = Dataset.from_list(grouped_ds)

    return grouped_ds

## データセット構築関数

In [11]:
from typing import Tuple, Dict, List, Optional
from datasets import Dataset
from transformers import PreTrainedTokenizerBase, BatchEncoding

def create_text_label_dataset(
    ds: Dataset,
    tokenizer: PreTrainedTokenizerBase,
    max_length: int,
    *,
    text_col: str = "text",
    label_col: str = "label",
    padding: bool | str = False,   # False / "longest" / "max_length"
    label2id: Optional[dict[str, int]] = None,
) -> Tuple[Dataset, dict[str, float]]:
    """(text, label) Dataset をシンプルにトークナイズして返す。

    - ds は少なくとも {text_col, label_col} を持つ想定（例: "text", "label"）
    - 返り値の Dataset は tokenized columns + labels を持つ
    - NER用の BIO 付与や span マッチは行わない

    Args:
        ds: 入力 Dataset（例: text, label, uuid, field など）
        tokenizer: Hugging Face tokenizer
        max_length: 最大長
        text_col: テキスト列名
        label_col: ラベル列名（"PROFILE"/"NONE" など）
        padding: False / "longest" / "max_length"
        label2id: ラベルをIDにしたい場合に渡す辞書（例 {"NONE":0,"PROFILE":1,"PII":2}）

    Returns:
        tokenized Dataset と簡単な統計 dict
    """
    # ちょい統計（任意）
    labels = ds[label_col]
    unique_labels = sorted(set(labels))
    stats: dict[str, float] = {
        "num_rows": float(len(ds)),
        "num_unique_labels": float(len(unique_labels)),
    }

    def _tokenize_batch(examples: dict[str, list]) -> dict[str, list]:
        texts: List[str] = examples[text_col]
        tokenized: BatchEncoding = tokenizer(
            texts,
            truncation=True,
            max_length=max_length,
            padding=padding,  # False / "longest" / "max_length"
        )

        out: dict[str, list] = dict(tokenized)
        if label2id is not None:
            out["labels"] = [label2id[lbl] for lbl in examples[label_col]]
        else:
            out["labels"] = examples[label_col]

        return out

    tokenized_ds = ds.map(
        _tokenize_batch,
        batched=True,
        desc="Tokenizing (text, label) dataset",
    )

    keep_cols = {"input_ids", "attention_mask", "token_type_ids", "labels"}
    remove_cols = [c for c in tokenized_ds.column_names if c not in keep_cols]
    if remove_cols:
        tokenized_ds = tokenized_ds.remove_columns(remove_cols)

    return tokenized_ds, stats

## メトリクス関数

In [12]:
from typing import Callable
import numpy as np
from transformers import EvalPrediction
from sklearn.metrics import f1_score

def make_compute_metrics(
    *,
    average: str = "macro",  # "macro" / "micro" / "weighted"
) -> Callable[[EvalPrediction], dict[str, float]]:
    """単一ラベル分類用メトリクス（accuracy + F1）"""

    def compute_metrics(eval_pred: EvalPrediction) -> dict[str, float]:
        preds = np.argmax(eval_pred.predictions, axis=-1)
        labels = eval_pred.label_ids

        acc = (preds == labels).mean().item()
        f1 = f1_score(labels, preds, average=average)

        return {
            "accuracy": acc,
            "f1": f1,
        }

    return compute_metrics

## wandb 認証・設定初期化

In [13]:
config = Config()
output_dir: str = f"{config.output_base_dir}/{config.exp}"


wandb.login("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")

wandb.init(
    project=config.wandb_project,
    name=f"exp{config.exp}",
    group=f"exp{config.exp}",
    config={
        "exp": config.exp,
        "model_name": config.model_name,
        "data_path": config.data_path,
        "max_length": config.max_length,
        "num_train_epochs": config.num_train_epochs,
        "per_device_train_batch_size": config.per_device_train_batch_size,
        "gradient_accumulation_steps": config.gradient_accumulation_steps,
        "learning_rate": config.learning_rate,
        "weight_decay": config.weight_decay,
        "warmup_ratio": config.warmup_ratio,
        "fp16": config.fp16,
        "eval_ratio": config.eval_ratio,
        "seed": config.seed,
        "early_stopping_patience": config.early_stopping_patience,
    },
)

logger.info("model_name: %s", config.model_name)
logger.info("data_path: %s", config.data_path)
logger.info("output_dir: %s", output_dir)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yohei-kobashi (weblab-llm-competition-2025-bridge) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## データセット読み込み・NER データセット構築

In [14]:
ds: Dataset = load_pseudo_dataset(config.data_path)
logger.info("texts: %d", len(ds))

tokenizer: PreTrainedTokenizerBase = AutoTokenizer.from_pretrained(
    config.model_name
)

ner_ds: Dataset
span_stats: dict[str, float]
ner_ds, span_stats = create_text_label_dataset(ds, tokenizer, config.max_length, label2id=LABEL2ID)
logger.info("NER dataset: %d examples", len(ner_ds))

Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.83M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/968 [00:00<?, ?B/s]

Tokenizing (text, label) dataset:   0%|          | 0/33119 [00:00<?, ? examples/s]

## Train / Eval 分割

In [15]:
split: DatasetDict = ner_ds.train_test_split(
    test_size=config.eval_ratio, seed=config.seed
)
train_ds: Dataset = split["train"]
eval_ds: Dataset = split["test"]
logger.info("train: %d, eval: %d", len(train_ds), len(eval_ds))

wandb.log(
    {
        "dataset/total_texts": len(ds),
        "dataset/ner_examples": len(ner_ds),
        "dataset/train_size": len(train_ds),
        "dataset/eval_size": len(eval_ds),
        **span_stats,
    }
)

## モデル・Trainer セットアップ

In [16]:
model: PreTrainedModel = AutoModelForSequenceClassification.from_pretrained(
    config.model_name,
    num_labels=len(LABEL_NAMES),
    label2id=LABEL2ID,
    id2label=ID2LABEL,
)

training_args: TrainingArguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=config.num_train_epochs,
    per_device_train_batch_size=config.per_device_train_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
    warmup_ratio=config.warmup_ratio,
    fp16=config.fp16,
    dataloader_num_workers=config.dataloader_num_workers,
    eval_strategy=config.eval_strategy,
    save_strategy=config.save_strategy,
    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    logging_steps=config.logging_steps,
    seed=config.seed,
    metric_for_best_model=config.metric_for_best_model,
    load_best_model_at_end=True,
    report_to=config.report_to,
)

data_collator: DataCollatorWithPadding = (
    DataCollatorWithPadding(tokenizer=tokenizer, padding=True)
)
compute_metrics: Callable[[EvalPrediction], dict[str, float]] = (
    make_compute_metrics()
)

early_stopping: EarlyStoppingCallback = EarlyStoppingCallback(
    early_stopping_patience=config.early_stopping_patience,
)

trainer: Trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    callbacks=[early_stopping],
)

model.safetensors:   0%|          | 0.00/530M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/118 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: sbintuitions/modernbert-ja-130m
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## 学習実行

In [17]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1
100,2.343595,0.426933,0.810085,0.856751
200,1.129329,0.785718,0.740338,0.788889
300,0.934150,0.379221,0.839372,0.881045
400,0.869298,0.514720,0.791365,0.841433
500,0.698391,0.442602,0.812802,0.856372
600,0.741452,0.392084,0.828200,0.871440
700,0.682101,0.279932,0.878019,0.909946
800,0.650509,0.294883,0.872886,0.904133
900,0.570620,0.290280,0.878623,0.909458
1000,0.593795,0.311136,0.880435,0.910458


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1864, training_loss=0.6949526045967069, metrics={'train_runtime': 529.0697, 'train_samples_per_second': 112.677, 'train_steps_per_second': 3.523, 'total_flos': 562992062157402.0, 'train_loss': 0.6949526045967069, 'epoch': 2.0})

## モデル保存・wandb 終了

In [18]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
logger.info("model saved to %s", output_dir)

wandb.finish()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

dataset/eval_size,▁
dataset/ner_examples,▁
dataset/total_texts,▁
dataset/train_size,▁
eval/accuracy,▄▁▅▃▄▅▇▇▇▇▇███████
eval/f1,▅▁▆▄▅▅▇▇▇▇▇███████
eval/loss,▃█▃▄▄▃▁▂▂▂▂▁▁▁▁▁▁▁
eval/runtime,▁▃▃▄▃▂▃▄▄▃▂▃█▃▅▃▂▅
eval/samples_per_second,█▅▆▅▅▇▆▅▅▆▇▆▁▅▄▆▆▄
eval/steps_per_second,█▅▆▅▅▇▆▅▅▆▇▆▁▅▄▆▆▄
+7,...


## Nemotron への疑似ラベル付与（Streaming + Resume + GPU Batch）

In [19]:
import json
import os
import re
from dataclasses import dataclass
from typing import Iterable

import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

PUNCT_SPLIT_RE = re.compile(r"[。、]+")
NEMOTRON_FIELDS = [
    "professional_persona",
    "sports_persona",
    "arts_persona",
    "travel_persona",
    "culinary_persona",
    "cultural_background",
]


def normalize_field(value) -> list[str]:
    if value is None:
        return []
    if isinstance(value, str):
        return [value]
    if isinstance(value, (list, tuple)):
        out = []
        for v in value:
            if v is None:
                continue
            out.append(v if isinstance(v, str) else str(v))
        return out
    return [str(value)]


def split_clauses(text: str) -> list[str]:
    return [c.strip() for c in PUNCT_SPLIT_RE.split(text) if c.strip()]


def load_processed_keys(output_path: str) -> set[tuple[str, str]]:
    processed: set[tuple[str, str]] = set()
    if not os.path.exists(output_path):
        return processed

    with open(output_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
                processed.add((str(row.get("uuid")), row.get("clause", "")))
            except json.JSONDecodeError:
                continue
    return processed


def infer_labels(
    clauses: list[str],
    model: AutoModelForSequenceClassification,
    tokenizer: AutoTokenizer,
    device: torch.device,
    batch_size: int,
    max_length: int,
) -> list[str]:
    labels: list[str] = []
    id2label = model.config.id2label

    for i in range(0, len(clauses), batch_size):
        batch_texts = clauses[i : i + batch_size]
        encoded = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=True,
        )
        encoded = {k: v.to(device, non_blocking=True) for k, v in encoded.items()}

        with torch.inference_mode():
            if device.type == "cuda":
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits = model(**encoded).logits
            else:
                logits = model(**encoded).logits

        pred_ids = torch.argmax(logits, dim=-1).detach().cpu().tolist()
        labels.extend(id2label[int(pid)] for pid in pred_ids)

    return labels


@dataclass(frozen=True)
class PseudoLabelConfig:
    dataset_name: str = "nvidia/Nemotron-Personas-Japan"
    split: str = "train"
    streaming: bool = True
    output_path: str = "drive/My Drive/Colab Notebooks/quasi_identifer/annotations.jsonl"
    model_path: str = output_dir
    max_rows: int = 0       # 0: 制限なし
    max_clauses: int = 0    # 0: 制限なし
    write_batch_size: int = 4096
    infer_batch_size: int = 512
    max_length: int = config.max_length


infer_cfg = PseudoLabelConfig()
processed_keys = load_processed_keys(infer_cfg.output_path)
print(f"resume: {len(processed_keys)} clauses already processed")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    free_mem, total_mem = torch.cuda.mem_get_info()
    total_gb = total_mem / (1024**3)
    if total_gb >= 70:
        infer_batch_size = max(infer_cfg.infer_batch_size, 1024)
    elif total_gb >= 40:
        infer_batch_size = max(infer_cfg.infer_batch_size, 512)
    elif total_gb >= 20:
        infer_batch_size = max(infer_cfg.infer_batch_size, 256)
    else:
        infer_batch_size = max(64, min(infer_cfg.infer_batch_size, 128))
else:
    infer_batch_size = min(infer_cfg.infer_batch_size, 64)

print(f"device={device}, infer_batch_size={infer_batch_size}")

label_tokenizer = AutoTokenizer.from_pretrained(infer_cfg.model_path)
label_model = AutoModelForSequenceClassification.from_pretrained(infer_cfg.model_path)
label_model.to(device)
label_model.eval()

try:
    stream_ds = load_dataset(
        infer_cfg.dataset_name,
        split=infer_cfg.split,
        streaming=infer_cfg.streaming,
    )
except Exception:
    ds_dict = load_dataset(infer_cfg.dataset_name, streaming=infer_cfg.streaming)
    available_splits = list(ds_dict.keys())
    use_split = available_splits[0]
    print(f"split '{infer_cfg.split}' not found. using '{use_split}'")
    stream_ds = ds_dict[use_split]

os.makedirs(os.path.dirname(infer_cfg.output_path) or ".", exist_ok=True)

pending_records: list[dict] = []
seen_clauses = 0
written_clauses = 0
skipped_clauses = 0

progress = tqdm(desc="Labeling clauses", unit="clause")

with open(infer_cfg.output_path, "a", encoding="utf-8") as out_f:
    for row_index, row in enumerate(stream_ds):
        if infer_cfg.max_rows > 0 and row_index >= infer_cfg.max_rows:
            break

        uuid = str(row.get("uuid"))
        for field in NEMOTRON_FIELDS:
            for text in normalize_field(row.get(field)):
                for clause in split_clauses(text):
                    seen_clauses += 1
                    if infer_cfg.max_clauses > 0 and seen_clauses > infer_cfg.max_clauses:
                        break

                    key = (uuid, clause)
                    if key in processed_keys:
                        skipped_clauses += 1
                        progress.update(1)
                        continue

                    pending_records.append(
                        {
                            "row_index": row_index,
                            "uuid": uuid,
                            "field": field,
                            "clause": clause,
                        }
                    )

                    if len(pending_records) >= infer_cfg.write_batch_size:
                        clauses = [r["clause"] for r in pending_records]
                        labels = infer_labels(
                            clauses=clauses,
                            model=label_model,
                            tokenizer=label_tokenizer,
                            device=device,
                            batch_size=infer_batch_size,
                            max_length=infer_cfg.max_length,
                        )
                        for rec, label in zip(pending_records, labels):
                            rec["label"] = label
                            out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                            processed_keys.add((rec["uuid"], rec["clause"]))
                            written_clauses += 1
                            progress.update(1)
                        out_f.flush()
                        pending_records.clear()

                if infer_cfg.max_clauses > 0 and seen_clauses > infer_cfg.max_clauses:
                    break
            if infer_cfg.max_clauses > 0 and seen_clauses > infer_cfg.max_clauses:
                break
        if infer_cfg.max_clauses > 0 and seen_clauses > infer_cfg.max_clauses:
            break

    if pending_records:
        clauses = [r["clause"] for r in pending_records]
        labels = infer_labels(
            clauses=clauses,
            model=label_model,
            tokenizer=label_tokenizer,
            device=device,
            batch_size=infer_batch_size,
            max_length=infer_cfg.max_length,
        )
        for rec, label in zip(pending_records, labels):
            rec["label"] = label
            out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            processed_keys.add((rec["uuid"], rec["clause"]))
            written_clauses += 1
            progress.update(1)
        out_f.flush()
        pending_records.clear()

progress.close()
print(
    f"done. seen={seen_clauses}, written={written_clauses}, skipped={skipped_clauses}, output={infer_cfg.output_path}"
)



resume: 34512 clauses already processed
device=cuda, infer_batch_size=1024


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


Loading weights:   0%|          | 0/120 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

Labeling clauses: 0clause [00:00, ?clause/s]

done. seen=31759869, written=31711138, skipped=48731, output=drive/My Drive/Colab Notebooks/quasi_identifer/annotations.jsonl
